# Raccolta Dati da Reddit — r/Italia, keyword: *notizie*
Raccolta tramite **Arctic Shift** (archivio pubblico Reddit per ricerca accademica).

**Obiettivo**: raccogliere **100 post** di r/Italia con "notizie" nel titolo e i **top 25 commenti** per ciascun post.

Il dataset permetterà di confrontare l'emozione espressa nel post con quella dei commenti tramite **ELIta**.
I modelli pre-addestrati (feel-it, BERT italiano) saranno usati in seguito come validazione esterna.

## Installazione dipendenze

In [1]:
# !pip install requests spacy tqdm
# !python -m spacy download it_core_news_sm

## Configurazione

In [2]:
import requests
import time
import calendar
import pandas as pd
from datetime import datetime

SUBREDDIT         = "Italia"
KEYWORD           = "notizie"
N_POSTS           = 200
COMMENTS_PER_POST = 25

OUTPUT_POSTS_CSV    = f"posts_{SUBREDDIT}_{KEYWORD}.csv"
OUTPUT_COMMENTS_CSV = f"comments_{SUBREDDIT}_{KEYWORD}.csv"

BASE_URL = "https://arctic-shift.photon-reddit.com/api"
HEADERS  = {"User-Agent": "python:elita.tesi.notizie:v1.0 (academic NLP)"}

print("Configurazione:")
print(f"  Subreddit         : r/{SUBREDDIT}")
print(f"  Keyword           : '{KEYWORD}'")
print(f"  Post da raccogliere   : {N_POSTS}")
print(f"  Commenti per post     : {COMMENTS_PER_POST}")

/Users/veronicabosso/PycharmProjects/ELIta_tesi/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Configurazione:
  Subreddit         : r/Italia
  Keyword           : 'notizie'
  Post da raccogliere   : 200
  Commenti per post     : 25


## Test connessione API
Attenzione, con servizi intermedi come **Arctic Shift** le risposte possono variare. Se lo status è 200 OK, altrimenti se è 422 riprova dopo qualche secondo.

In [3]:
# Test 1: endpoint posts/search
resp = requests.get(f"{BASE_URL}/posts/search", headers=HEADERS, timeout=20, params={
    "subreddit": SUBREDDIT, "title": KEYWORD, "limit": 2, "after": "2025-01-01", "before": "2025-01-31"
})
print(f"[posts/search]    Status: {resp.status_code}")
if resp.status_code == 200 and resp.json().get("data"):
    p = resp.json()["data"][0]
    sample_post_id = p.get("id", "")
    print(f"  Titolo   : {p.get('title','')[:80]}")
    print(f"  Score    : {p.get('score',0)}  |  Commenti: {p.get('num_comments',0)}")
    print(f"  Post ID  : {sample_post_id}")
else:
    sample_post_id = ""
    print(f"  Errore: {resp.text[:200]}")

# Test 2: endpoint comments/search per un post specifico
if sample_post_id:
    resp2 = requests.get(f"{BASE_URL}/comments/search", headers=HEADERS, timeout=20, params={
        "link_id": sample_post_id, "limit": 3
    })
    print(f"\n[comments/search] Status: {resp2.status_code}")
    if resp2.status_code == 200 and resp2.json().get("data"):
        print(f"  Primo commento: {resp2.json()['data'][0].get('body','')[:150]}")
    else:
        print(f"  Errore: {resp2.text[:200]}")

[posts/search]    Status: 200
  Titolo   : Il Pil dell'Italia è fermo, la disoccupazione sale al 6,2% - Notizie
  Score    : 241  |  Commenti: 120
  Post ID  : 1idmjsb

[comments/search] Status: 200
  Primo commento: [removed]


## Raccolta Post
Cerchiamo i post di r/Italia con "notizie" nel **titolo** tramite `/api/posts/search`.
Finestre mensili per evitare il rate limit (stessa tecnica dei commenti).

In [4]:
def collect_posts(subreddit, keyword, target=100):
    all_posts = []

    # Finestre mensili: 2025 → 2024 → 2023 (ci fermiamo appena raggiungiamo il target)
    time_windows = []
    for year in [2025, 2024, 2023]:
        max_month = 5 if year == 2025 else 12
        for month in range(max_month, 0, -1):
            last_day = calendar.monthrange(year, month)[1]
            time_windows.append((f"{year}-{month:02d}-01", f"{year}-{month:02d}-{last_day}", f"{year}-{month:02d}"))

    for after, before, label in time_windows:
        if len(all_posts) >= target:
            break

        params = {"subreddit": subreddit, "title": keyword, "limit": 25, "after": after, "before": before}
        last_utc = None

        while len(all_posts) < target:
            if last_utc:
                params["before"] = last_utc
            try:
                resp = requests.get(f"{BASE_URL}/posts/search", headers=HEADERS, params=params, timeout=30)
                if resp.status_code == 422:
                    break
                resp.raise_for_status()
                items = resp.json().get("data", [])
            except Exception as e:
                print(f"  Errore {label}: {e}")
                break

            if not items:
                break

            for p in items:
                if len(all_posts) >= target:
                    break
                title = p.get("title", "").strip()
                if not title:
                    continue
                selftext = p.get("selftext", "").strip()
                if selftext in ("[removed]", "[deleted]"):
                    selftext = ""
                ts = float(p.get("created_utc", 0))
                all_posts.append({
                    "post_id"     : p.get("id", ""),
                    "title"       : title,
                    "selftext"    : selftext,
                    "author"      : p.get("author", ""),
                    "timestamp"   : datetime.utcfromtimestamp(ts).isoformat(),
                    "score"       : p.get("score", 0),
                    "num_comments": p.get("num_comments", 0),
                    "url"         : "https://reddit.com" + p.get("permalink", "") if p.get("permalink") else "",
                })

            oldest_utc = min(float(p.get("created_utc", 0)) for p in items)
            last_utc   = datetime.utcfromtimestamp(oldest_utc).strftime("%Y-%m-%dT%H:%M:%S")
            if len(items) < 25:
                break
            time.sleep(1)

        print(f"  {label}: {len(all_posts)} post totali", end="\r")
        time.sleep(1.5)

    print(f"\nPost raccolti: {len(all_posts)}")
    return all_posts

posts = collect_posts(SUBREDDIT, KEYWORD, target=N_POSTS)
df_posts = pd.DataFrame(posts)
print(df_posts[["post_id", "title", "score", "num_comments", "timestamp"]].head(5))

  2023-11: 200 post totali
Post raccolti: 200
   post_id                                              title  score  \
0  1kwlwc6                Quanto odio quei siti di notizie...     31   
1  1kuc88y                          Notizie da un macchinista      1   
2  1kq6axq  Secondo voi con l'aumento dell'immigrazione cl...      0   
3  1kna8hg  Un sms della cugina di Chiara Poggi: 'Mi sa ch...     48   
4  1kn9cp6  Su Reddit ma anche sui quotidiani spesso vedo ...    126   

   num_comments            timestamp  
0            24  2025-05-27T12:23:55  
1             1  2025-05-24T13:57:01  
2            26  2025-05-19T08:07:57  
3             9  2025-05-15T14:54:26  
4            57  2025-05-15T14:17:43  


## Raccolta Commenti per Post
Per ogni post, raccogliamo fino a **100 commenti** dall'API (ordinati per `created_utc` discendente),
poi li ordiniamo per **score decrescente** lato Python e teniamo i **top 25**.

Questo garantisce che i commenti selezionati siano quelli più apprezzati dalla community,
che rappresentano meglio la reazione collettiva al post.

In [9]:
### ATTENZIONE non esuire sta cella se non vuoi perdere 5 min di vita
def collect_comments_for_posts(df_posts, n_comments=25, fetch_limit=100):
    """
    Per ogni post:
      - recupera fino a `fetch_limit` commenti dall'API paginando correttamente
      - ordina per score decrescente
      - tiene i top `n_comments`
    """
    all_comments = []

    for i, row in df_posts.iterrows():
        post_id  = row["post_id"]
        raw_items = []
        last_utc  = None

        # Paginazione corretta: continua finché non raggiungiamo fetch_limit
        # o l'API non ha più risultati
        while len(raw_items) < fetch_limit:
            params = {"link_id": post_id, "limit": 100}
            if last_utc:
                params["before"] = last_utc

            try:
                resp = requests.get(f"{BASE_URL}/comments/search", headers=HEADERS,
                                    params=params, timeout=30)
                if resp.status_code != 200:
                    break
                items = resp.json().get("data", [])
            except Exception as e:
                print(f"  Errore post {post_id}: {e}")
                break

            if not items:
                break

            raw_items.extend(items)

            # Aggiorna il cursore PRIMA di decidere se continuare
            oldest_utc = min(float(c.get("created_utc", 0)) for c in items)
            last_utc   = datetime.utcfromtimestamp(oldest_utc).strftime("%Y-%m-%dT%H:%M:%S")

            # Se l'API ha restituito meno di 100, non ci sono altre pagine
            if len(items) < 100:
                break

            time.sleep(0.3)

        # Filtra rimossi/troppo corti, ordina per score, tieni i top n
        valid = [
            c for c in raw_items
            if c.get("body", "").strip() not in ("", "[deleted]", "[removed]")
            and len(c.get("body", "").strip()) >= 10
        ]
        top = sorted(valid, key=lambda c: c.get("score", 0), reverse=True)[:n_comments]

        for rank, c in enumerate(top, start=1):
            ts = float(c.get("created_utc", 0))
            all_comments.append({
                "comment_id"   : c.get("id", ""),
                "post_id"      : post_id,
                "rank_by_score": rank,
                "comment_text" : c.get("body", "").strip(),
                "author"       : c.get("author", ""),
                "timestamp"    : datetime.utcfromtimestamp(ts).isoformat(),
                "score"        : c.get("score", 0),
                "permalink"    : "https://reddit.com" + c.get("permalink", "")
                                  if c.get("permalink") else "",
            })

        print(f"  [{i+1:3d}/{len(df_posts)}] {post_id}: "
              f"{len(raw_items)} recuperati → {len(valid)} validi → top {len(top)}")
        time.sleep(0.8)

    print(f"\nCommenti totali: {len(all_comments)}")
    return all_comments

comments = collect_comments_for_posts(df_posts, n_comments=COMMENTS_PER_POST, fetch_limit=100)
df_comments = pd.DataFrame(comments)
print(df_comments[["comment_id", "post_id", "rank_by_score", "score", "comment_text"]].head(8))

  [  1/200] 1kwlwc6: 26 recuperati → 25 validi → top 25
  [  2/200] 1kuc88y: 1 recuperati → 1 validi → top 1
  [  3/200] 1kq6axq: 33 recuperati → 31 validi → top 25
  [  4/200] 1kna8hg: 10 recuperati → 10 validi → top 10
  [  5/200] 1kn9cp6: 59 recuperati → 58 validi → top 25
  [  6/200] 1kjv1dn: 1 recuperati → 1 validi → top 1
  [  7/200] 1khu4t1: 0 recuperati → 0 validi → top 0
  [  8/200] 1khu16n: 0 recuperati → 0 validi → top 0
  [  9/200] 1khu0wh: 0 recuperati → 0 validi → top 0
  [ 10/200] 1khu0sz: 0 recuperati → 0 validi → top 0
  [ 11/200] 1kem3wd: 1 recuperati → 1 validi → top 1
  [ 12/200] 1ke1jvk: 1 recuperati → 1 validi → top 1
  [ 13/200] 1kd66x8: 5 recuperati → 4 validi → top 4
  [ 14/200] 1kczbl2: 5 recuperati → 5 validi → top 5
  [ 15/200] 1kcz8yh: 56 recuperati → 54 validi → top 25
  [ 16/200] 1kc8tre: 6 recuperati → 6 validi → top 6
  [ 17/200] 1ka1u9n: 3 recuperati → 3 validi → top 3
  [ 18/200] 1k9s902: 1 recuperati → 1 validi → top 1
  [ 19/200] 1k7hecq: 3 recupera

## Salvataggio CSV
Quattro file:
- **`posts_Italia_notizie.csv`** — un post per riga
- **`comments_Italia_notizie.csv`** — un commento per riga, con `post_id` come chiave di join
- **`merged_Italia_notizie.csv`** — ogni commento affiancato ai dati del post di origine
- **`corpus_Italia_notizie.csv`** — long format: una riga per documento (post o commento), colonna `type` per distinguerli, `text` come testo da analizzare con ELIta

In [10]:
# Salva posts
df_posts.to_csv(OUTPUT_POSTS_CSV, index=False, encoding="utf-8-sig")
print(f"Post salvati    : {len(df_posts)} righe → '{OUTPUT_POSTS_CSV}'")

# Salva commenti
df_comments.to_csv(OUTPUT_COMMENTS_CSV, index=False, encoding="utf-8-sig")
print(f"Commenti salvati: {len(df_comments)} righe → '{OUTPUT_COMMENTS_CSV}'")

# Merged: ogni commento affiancato ai dati del suo post (left join)
OUTPUT_MERGED_CSV = f"merged_{SUBREDDIT}_{KEYWORD}.csv"
df_merged = df_comments.merge(
    df_posts[["post_id", "title", "selftext", "score", "num_comments", "timestamp", "url"]],
    on="post_id",
    how="left",
    suffixes=("_comment", "_post")
)
df_merged.to_csv(OUTPUT_MERGED_CSV, index=False, encoding="utf-8-sig")
print(f"Merged salvato  : {len(df_merged)} righe → '{OUTPUT_MERGED_CSV}'")

# Corpus long-format: una riga per documento (post o commento)
OUTPUT_CORPUS_CSV = f"corpus_{SUBREDDIT}_{KEYWORD}.csv"

posts_long = df_posts.assign(
    type="post",
    doc_id=df_posts["post_id"],
    text=df_posts.apply(
        lambda r: (r["title"] + "\n" + r["selftext"]).strip() if r["selftext"] else r["title"],
        axis=1
    ),
    rank_by_score=None,
)[["doc_id", "post_id", "type", "text", "author", "timestamp", "score", "rank_by_score"]]

comments_long = df_comments.rename(columns={"comment_id": "doc_id", "comment_text": "text"}) \
    .assign(type="comment")[["doc_id", "post_id", "type", "text", "author", "timestamp", "score", "rank_by_score"]]

df_corpus = pd.concat([posts_long, comments_long], ignore_index=True)
df_corpus.to_csv(OUTPUT_CORPUS_CSV, index=False, encoding="utf-8-sig")
print(f"Corpus salvato  : {len(df_corpus)} righe → '{OUTPUT_CORPUS_CSV}'")
print(f"  post: {(df_corpus['type']=='post').sum()}  |  commenti: {(df_corpus['type']=='comment').sum()}")

# Anteprima
print("\n--- Corpus: prime 2 righe (post) ---")
display(df_corpus[["doc_id", "post_id", "type", "text", "score"]].head(2))
print("\n--- Corpus: ultimi 2 righe (commenti) ---")
display(df_corpus[["doc_id", "post_id", "type", "text", "score"]].tail(2))

Post salvati    : 200 righe → 'posts_Italia_notizie.csv'
Commenti salvati: 2320 righe → 'comments_Italia_notizie.csv'
Merged salvato  : 2320 righe → 'merged_Italia_notizie.csv'
Corpus salvato  : 2520 righe → 'corpus_Italia_notizie.csv'
  post: 200  |  commenti: 2320

--- Corpus: prime 2 righe (post) ---


,doc_id,post_id,type,text,score
0,1kwlwc6,1kwlwc6,post,Quanto odio quei siti di notizie...\n...in cui...,31
1,1kuc88y,1kuc88y,post,Notizie da un macchinista\nSe nei prossimi ved...,1



--- Corpus: ultimi 2 righe (commenti) ---


,doc_id,post_id,type,text,score
2518,kaapuhv,1817buy,comment,Ciao sono in Botswana e sto mangiando una pizz...,4
2519,kaan9yn,1817buy,comment,"Strano però, ma questi non erano quelli ossess...",2


## Statistiche del corpus

In [11]:
import plotly.express as px

print("=" * 50)
print("STATISTICHE CORPUS")
print("=" * 50)
print(f"Post totali              : {len(df_posts)}")
print(f"  - con testo (selftext) : {(df_posts['selftext'] != '').sum()}")
print(f"  - solo link            : {(df_posts['selftext'] == '').sum()}")
print(f"Commenti totali          : {len(df_comments)}")
print(f"Media commenti per post  : {len(df_comments)/len(df_posts):.1f}")
print(f"Autori unici (post)      : {df_posts['author'].nunique()}")
print(f"Autori unici (commenti)  : {df_comments['author'].nunique()}")
print(f"Periodo post             : {df_posts['timestamp'].min()[:10]} → {df_posts['timestamp'].max()[:10]}")

# Post per mese
df_posts["mese"] = df_posts["timestamp"].str[:7]
fig1 = px.bar(df_posts["mese"].value_counts().sort_index().reset_index(),
              x="mese", y="count", title="Post per mese",
              labels={"mese": "Mese", "count": "Post"})
fig1.show()

# Distribuzione commenti per post
commenti_per_post = df_comments.groupby("post_id").size().reset_index(name="n_commenti")
fig2 = px.histogram(commenti_per_post, x="n_commenti", nbins=26,
                    title="Distribuzione commenti raccolti per post",
                    labels={"n_commenti": "Commenti raccolti", "count": "Post"})
fig2.show()

# Top 10 post per score
print("\nTop 10 post per score:")
display(df_posts.nlargest(10, "score")[["title", "score", "num_comments", "timestamp"]])

STATISTICHE CORPUS
Post totali              : 200
  - con testo (selftext) : 44
  - solo link            : 156
Commenti totali          : 2320
Media commenti per post  : 11.6
Autori unici (post)      : 112
Autori unici (commenti)  : 1247
Periodo post             : 2023-11-11 → 2025-05-27



Top 10 post per score:


,title,score,num_comments,timestamp
111,"basta politica e brutte notizie, guardate sta ...",912,138,2024-04-22T14:06:33
152,Classe va a vedere uno spettacolo sull'inclusi...,344,72,2024-02-08T09:49:08
107,Buone notizie Telepass decide di modificare og...,314,153,2024-05-07T11:29:03
14,Notizie top tier,262,54,2025-05-02T12:03:46
42,"Il Pil dell'Italia è fermo, la disoccupazione ...",241,120,2025-01-30T13:07:36
23,Le grandi notizie di Repubblica,211,25,2025-03-16T23:13:06
153,"Preside preso a pugni, le sue parole: ""I genit...",174,175,2024-02-08T07:13:03
53,Studenti italiani campioni europei dei compiti...,171,50,2024-12-06T22:04:26
21,"Nordio: ""Alcune etnie hanno sensibilità divers...",170,318,2025-04-03T09:37:58
150,Gli alunni non sanno più scrivere in corsivo. ...,167,376,2024-02-09T11:59:49


## Tokenizzazione, Lemmatizzazione e POS-tagging

Utilizziamo il modello italiano `it_core_news_sm` di spaCy per analizzare tutti i documenti del corpus (post e commenti). Per ogni token estraiamo:
- `token`: testo originale
- `lemma`: forma base in minuscolo
- `pos`: parte del discorso (NOUN, VERB, ADJ, …)
- `is_adj`: flag booleano se è un aggettivo

Filtriamo spazi e punteggiatura. L'identificatore è `doc_id` (che vale sia per post che commenti).

Salviamo il risultato in `tokens_Italia_notizie.csv`.

In [12]:
import spacy
import emoji as emoji_lib
from tqdm.auto import tqdm

nlp = spacy.load("it_core_news_sm")

def process_text(text):
    doc = nlp(str(text))
    tokens = []
    for token in doc:
        if token.is_space or token.is_punct:
            continue
        # Forza POS uniforme per tutte le emoji, indipendentemente da spaCy
        pos = "EMOJI" if emoji_lib.emoji_count(token.text) > 0 else token.pos_
        tokens.append({
            "token" : token.text,
            "lemma" : token.lemma_.lower(),
            "pos"   : pos,
            "is_adj": pos == "ADJ",
        })
    return tokens

# Test su un documento di esempio (primo post)
esempio_row = df_corpus.iloc[0]
print(f"Tipo: {esempio_row['type']}  |  doc_id: {esempio_row['doc_id']}")
print(f"Testo:\n{esempio_row['text'][:200]}\n")
print("Analisi (primi 15 token):")
for t in process_text(esempio_row["text"])[:15]:
    flag = " <- ADJ" if t["is_adj"] else ""
    print(f"  {t['token']:20s} | {t['lemma']:20s} | {t['pos']}{flag}")

print(f"\nProcessamento di {len(df_corpus)} documenti con spaCy...")

all_tokens = []
for _, row in tqdm(df_corpus.iterrows(), total=len(df_corpus)):
    for t in process_text(row["text"]):
        all_tokens.append({"doc_id": row["doc_id"], **t})

df_tokens = pd.DataFrame(all_tokens)
df_adj    = df_tokens[df_tokens["is_adj"]]
df_emoji  = df_tokens[df_tokens["pos"] == "EMOJI"]

print(f"\nToken totali     : {len(df_tokens)}")
print(f"Aggettivi trovati: {len(df_adj)}")
print(f"Emoji trovate    : {len(df_emoji)} ({df_emoji['lemma'].nunique()} distinte)")

tokens_file = f"tokens_{SUBREDDIT}_{KEYWORD}.csv"
df_tokens.to_csv(tokens_file, index=False, encoding="utf-8-sig")
print(f"\nToken salvati in '{tokens_file}'")

Tipo: post  |  doc_id: 1kwlwc6
Testo:
Quanto odio quei siti di notizie...
...in cui ti chiedono di accettare 200 biscotti 🍪 🍪 per poter visualizzare l'articolo e non appena li accetti ti comunicano che comunque è a pagamento quindi sticaz

Analisi (primi 15 token):
  Quanto               | quanto               | ADV
  odio                 | odiare               | VERB
  quei                 | quello               | DET
  siti                 | sito                 | NOUN
  di                   | di                   | ADP
  notizie              | notizia              | NOUN
  in                   | in                   | ADP
  cui                  | cui                  | PRON
  ti                   | ti                   | PRON
  chiedono             | chiedere             | VERB
  di                   | di                   | ADP
  accettare            | accettare            | VERB
  200                  | 200                  | NUM
  biscotti             | biscotto             | NOU

100%|██████████| 2520/2520 [00:18<00:00, 135.39it/s]



Token totali     : 84867
Aggettivi trovati: 5502
Emoji trovate    : 112 (56 distinte)

Token salvati in 'tokens_Italia_notizie.csv'
